# MART_P03_PERMANENCIA_ESPECIALIDADE

**P03. Qual é o tempo médio de permanência (em dias) em leito hospitalar por
especialidade?** *(dashboard)*

Média de permanência (regra do TabNet) = soma dos dias de permanência ÷ quantidade de
internações, por ano da saída do paciente.

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO MART_P03_PERMANENCIA_ESPECIALIDADE
    (NR_ANO, NOME_ESPECIALIDADE, QTD_INTERNACOES, QTD_DIAS_PERMANENCIA, MEDIA_DIAS_PERMANENCIA)
SELECT t.NR_ANO,
       e.NOME_ESPECIALIDADE,
       SUM(f.QTD_INTERNACAO),
       SUM(f.QTD_DIAS_PERMANENCIA),
       ROUND(SUM(f.QTD_DIAS_PERMANENCIA) / NULLIF(SUM(f.QTD_INTERNACAO), 0), 2)
FROM FATO_INTERNACAO f
JOIN DIM_TEMPO t         ON t.SK_TEMPO = f.SK_TEMPO_SAIDA
JOIN DIM_ESPECIALIDADE e ON e.SK_ESPECIALIDADE = f.SK_ESPECIALIDADE
WHERE t.NR_ANO IN (SELECT DISTINCT TO_NUMBER(SUBSTR(ANO_MES_CMPT, 1, 4)) FROM FATO_INTERNACAO)
GROUP BY t.NR_ANO, e.NOME_ESPECIALIDADE
"""

In [ ]:
recarregar("MART_P03_PERMANENCIA_ESPECIALIDADE", SQL, origem="FATO_INTERNACAO")

In [ ]:
consultar("SELECT * FROM MART_P03_PERMANENCIA_ESPECIALIDADE FETCH FIRST 10 ROWS ONLY")